# 00.02 · Verify your environment (no installs yet)

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 45–75 min |
| 📋 Prerequisites | Lesson 00.01. Anaconda (or Miniconda), Jupyter, and Git for Windows (Git Bash) already installed. |
| 🔑 Keys | None |
| 🧠 Kernel | Run it in your **current** Jupyter kernel (usually Anaconda's `base`). That is the point: we check what you have. |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

1. **Check** the versions of Python, conda, pip, Jupyter, and git from **Git Bash** (and know the macOS/Linux equivalents).
2. **Make `conda` available in Git Bash**, for one session or permanently.
3. **Find out** which Python interpreter a notebook's kernel really uses (`sys.executable`), and why it can differ from the terminal's.
4. **Tell apart** the conda `base` environment, a named project environment, and a venv.
5. **Decide** your next step: create the course environment `lc-mastery` with the [create-course-env guide](00-02-create-course-env.md).

## 📚 Official reading (verified 2026-10-11)

| Page | Why |
|---|---|
| [Install LangChain](https://docs.langchain.com/oss/python/langchain/install) | LangChain v1 requires **Python 3.10+** |
| [LangChain v1 migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1) | "Dropped Python 3.9 support" section |
| [Python versions status](https://devguide.python.org/versions/) | Which Python versions still get fixes |
| [Conda environments (Anaconda)](https://www.anaconda.com/docs/getting-started/working-with-conda/environments) · [Managing environments (conda)](https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html) | Environments, `base`, activation |
| [`conda init`](https://docs.conda.io/projects/conda/en/latest/commands/init.html) | How conda hooks into shells such as Git Bash |
| [Installing the IPython kernel](https://ipython.readthedocs.io/en/stable/install/kernel_install.html) | One kernel per environment |
| Course guides: [VERIFY_ENVIRONMENT.md](../../setup/VERIFY_ENVIRONMENT.md) · [Python environments, compared](../../setup/PYTHON_ENVIRONMENTS_COMPARED.md) · [uv](https://docs.astral.sh/uv/) | Terminal commands, and the conda vs uv choice |

---
## Part A · In the terminal (Git Bash) first

**Do this before running the code cells.** Open [setup/VERIFY_ENVIRONMENT.md](../../setup/VERIFY_ENVIRONMENT.md) and follow **Option A (Git Bash)** from top to bottom. It takes about 10 minutes and changes nothing on your machine. In short:

```bash
# Git Bash (Windows) · Terminal (macOS/Linux)
python --version
which -a python                         # every python on the PATH, in order
source ~/anaconda3/etc/profile.d/conda.sh   # Windows/Git Bash: make conda available in THIS session
conda --version && conda env list
python -m pip --version
jupyter --version
git --version
```

Keep that terminal open: at the end of this notebook you will compare its answers with the notebook's.

> **Why Git Bash?** It is the shell that comes with Git for Windows, it uses the same commands as macOS and Linux (`ls`, `cp`, `which`, `source`), and it is what you will use to commit your work. The course uses it everywhere. Anaconda Prompt still works as a fallback.

---
## Part B · In the notebook: which Python is running me?

A notebook does not run code itself. It sends each cell to a **kernel**, a separate Python process. Which Python that is depends on the kernel you picked, **not** on the terminal you started Jupyter from. Let's ask the kernel directly.

In [ ]:
# Step 1: Basic facts about the interpreter running THIS notebook.
import platform
import sys
from pathlib import Path

print("Python version :", platform.python_version())
print("Executable     :", sys.executable)   # The python.exe/python binary of this kernel.
print("sys.prefix     :", sys.prefix)       # The environment folder it belongs to.
print("Platform       :", platform.platform())

**You should now see** something like this on Windows with Anaconda:

```text
Python version : 3.13.5
Executable     : C:\Users\<you>\anaconda3\python.exe
sys.prefix     : C:\Users\<you>\anaconda3
Platform       : Windows-11-10.0.26100-SP0
```

Note that Python always prints **Windows-style paths** (`C:\Users\...`) even if you started Jupyter from Git Bash, which writes the same folder as `/c/Users/...`. Same place, two spellings (see the table at the end of this notebook).

In [ ]:
# Step 2: Is this Python new enough for the course?
# VERIFIED 2026-10-11: LangChain v1 needs Python >= 3.10 (docs.langchain.com/oss/python/langchain/install).
# deepagents and ipykernel 7.x declare >= 3.11 on PyPI, so the course minimum is 3.11.
COURSE_MIN_PYTHON = (3, 11)

if sys.version_info >= COURSE_MIN_PYTHON:
    print(f"✅ Python {platform.python_version()} meets the course minimum (3.11+).")
else:
    print(f"❌ Python {platform.python_version()} is too old for this course (need 3.11+).")
    print("   Don't upgrade base: the course env you create next uses Python 3.13.")

In [ ]:
# Step 3: What KIND of environment is this? (conda base, conda env, venv, or system Python)
import os


def describe_environment(prefix: Path, base_prefix: Path) -> str:
    """Classify the environment that owns this interpreter.

    Args:
        prefix: ``sys.prefix`` (the active environment folder).
        base_prefix: ``sys.base_prefix`` (differs from prefix inside a venv).

    Returns:
        A one-line, human-readable description.
    """
    if prefix.parent.name == "envs" and (prefix / "conda-meta").is_dir():
        return f"conda environment '{prefix.name}' (a named env inside {prefix.parent.parent.name})"
    if (prefix / "conda-meta").is_dir():
        return "conda BASE environment (the one Anaconda installed)"
    if prefix != base_prefix:
        return "virtual environment (venv/virtualenv/uv), not conda"
    return "a Python installation that is not managed by conda or a venv"


print("Environment     :", describe_environment(Path(sys.prefix), Path(sys.base_prefix)))
# conda sets these when an env is activated in a terminal. They may be empty
# if Jupyter was started from the Start menu or Anaconda Navigator; that's fine.
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "<not set>"))
print("CONDA_PREFIX     :", os.environ.get("CONDA_PREFIX", "<not set>"))

### What just happened?

- `describe_environment` used two facts: conda environments contain a `conda-meta` folder, and named conda envs live in `.../anaconda3/envs/<name>`. A venv is detected because `sys.prefix` differs from `sys.base_prefix`.
- Most learners see **"conda BASE environment"** here. That is expected today, and it is exactly why the next step creates a separate course environment: you should never install course packages into `base` ([comparison guide](../../setup/PYTHON_ENVIRONMENTS_COMPARED.md#recommendation-for-this-course)).

---
## Part C · Which packages does this kernel see?

`importlib.metadata` reads the version of an installed package **without importing it** (importing can be slow or have side effects). A missing package is reported as "not installed" instead of crashing the cell.

In [ ]:
from importlib import metadata

# Package names as they appear on PyPI (pip install <name>).
PACKAGES = [
    "pip", "notebook", "jupyterlab", "ipykernel", "jupyter_client",
    "python-dotenv", "httpx", "pandas",
    "langchain", "langchain-core", "langgraph", "langchain-groq", "langsmith",
]


def installed_version(name: str) -> str:
    """Return the installed version of `name`, or 'not installed'."""
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"


for name in PACKAGES:
    print(f"{name:<16} {installed_version(name)}")

**You should now see** `pip`, `notebook` (7.x) or `jupyterlab` (4.x), `ipykernel`, and probably `pandas` with versions (Anaconda ships them in `base`), and **"not installed"** for the LangChain packages. That is correct: LangChain packages go into the course environment, not into `base`.

> Anaconda's `base` may already include `python-dotenv` or `httpx`. Don't rely on that: the course environment installs its own copies.

---
## Part D · Command-line tools, checked from Python

You may know the `!command` syntax in notebooks (for example `!git --version`). Be careful with it on Windows: **`!` runs the command in `cmd.exe`, not in Git Bash**, even if you started Jupyter from Git Bash. Bash-only syntax such as `source` or `~/...` fails there.

The portable way is Python's own `shutil.which` (find a program on the PATH) and `subprocess.run` (run it). It works the same on Windows, macOS, and Linux.

In [ ]:
import shutil
import subprocess

TOOLS = ["python", "conda", "pip", "jupyter", "git", "uv", "docker"]


def tool_version(tool: str, timeout_s: float = 15.0) -> str:
    """Return the first line of `<tool> --version`, or a friendly reason it is unavailable.

    Args:
        tool: Program name to look up on the PATH.
        timeout_s: Seconds to wait before giving up (some tools are slow to start).
    """
    path = shutil.which(tool)
    if path is None:
        return "not on this kernel's PATH"
    try:
        result = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=timeout_s)
    except (OSError, subprocess.TimeoutExpired) as exc:
        return f"found at {path}, but '--version' failed ({type(exc).__name__})"
    output = (result.stdout or result.stderr).strip().splitlines()
    return f"{output[0] if output else '(no output)'}   [{path}]"


for tool in TOOLS:
    print(f"{tool:<8} {tool_version(tool)}")

### What just happened?

- Each tool was looked up on the **kernel's** PATH. If you started Jupyter from Anaconda Navigator or the Start menu, that PATH can differ from your Git Bash PATH. So `conda` may show "not on this kernel's PATH" here even though it works in Git Bash. That's fine.
- `uv` and `docker` are optional. "not on this kernel's PATH" is fine for both.
- `git` must be found **somewhere** (here or in Git Bash), because the course is a git repository.

### One more environment check: the certificate bundle

Every HTTPS call (to Groq, LangSmith, PyPI...) checks the server's certificate against a **CA bundle**, a file of trusted certificate authorities. The variable `SSL_CERT_FILE` can point Python to a specific bundle. If it points to a file that **does not exist**, every API client crashes with `FileNotFoundError: [Errno 2] No such file or directory`.

On Windows + Git Bash, Anaconda's `openssl` package currently sets this variable to a wrong path (missing `Library`). The cell below checks it. It only reads the variable; it changes nothing.

In [ ]:
import os
from pathlib import Path

for var, must_be in [("SSL_CERT_FILE", "file"), ("SSL_CERT_DIR", "folder")]:
    value = os.environ.get(var, "")
    if not value:
        print(f"✅ {var} is not set: Python uses its default bundle (certifi for the API SDKs).")
    elif (Path(value).is_file() if must_be == "file" else Path(value).is_dir()):
        print(f"✅ {var} points to an existing {must_be}: {value}")
    else:
        print(f"❌ {var} points to a {must_be} that does NOT exist: {value}")
        print("   Fix it in Git Bash, with the course env active (create-course-env guide, step 4b):")
        print('   conda env config vars set SSL_CERT_FILE="$(cygpath -m "$CONDA_PREFIX")/Library/ssl/cacert.pem"')
        print("   conda deactivate && conda activate lc-mastery")

**You should now see** two ✅ lines. A ❌ here, in the `base` kernel, is not urgent (you won't call APIs from `base`), but the course kernel must show ✅: [step 4b of the create-course-env guide](00-02-create-course-env.md#step-4b) fixes it.

---
## Part E · Kernels Jupyter knows about

A **kernelspec** is a small JSON file that tells Jupyter how to start a kernel: which Python to run. `jupyter_client` (installed with Jupyter) can list them.

In [ ]:
try:
    from jupyter_client.kernelspec import KernelSpecManager
except ImportError:
    print("jupyter_client is not installed in this kernel's environment (unusual). Run `jupyter kernelspec list` in Git Bash instead.")
else:
    specs = KernelSpecManager().find_kernel_specs()  # {kernel name: folder of its kernel.json}
    for kernel_name, folder in sorted(specs.items()):
        print(f"{kernel_name:<20} {folder}")
    if "lc-mastery" in specs:
        print("\n✅ The course kernel 'lc-mastery' is registered.")
    else:
        print("\n➡️ No 'lc-mastery' kernel yet. You will create it in the create-course-env guide.")

---
## Part F · `%pip` vs `pip`: the #1 beginner trap

`pip` in a terminal installs into **whatever Python comes first on that terminal's PATH**. The `%pip` magic in a notebook always installs into **the kernel's Python** (it runs `sys.executable -m pip`). If the two differ, a package you installed "successfully" in the terminal is invisible to your notebook.

Run the cell, then compare it with `python -m pip --version` in Git Bash.

In [ ]:
# %pip is an IPython "magic" command. It targets the kernel's environment (sys.executable).
%pip --version

**You should now see** a line like `pip 25.x from C:\Users\<you>\anaconda3\Lib\site-packages\pip (python 3.13)`. Compare the folder with the terminal's answer:

| Notebook (`%pip --version`) vs Git Bash (`python -m pip --version`) | Meaning |
|---|---|
| Same folder | Terminal and notebook share one environment. |
| Different folders | Two environments. From now on, install course packages **from the notebook with `%pip`** or from a terminal where the course env is **activated**. |

Docs: [IPython magics (`%pip`)](https://ipython.readthedocs.io/en/stable/interactive/magics.html).

---
## Part G · Record your results

The next cell writes a ready-to-paste summary. Copy it into [`docs/research-log.md`](../../docs/research-log.md) (open the file in Jupyter's file browser or your editor). It contains **no secrets**, only versions and paths.

In [ ]:
from datetime import date

summary = f"""### Environment check · {date.today().isoformat()}
- Python:          {platform.python_version()} ({sys.executable})
- Environment:     {describe_environment(Path(sys.prefix), Path(sys.base_prefix))}
- pip:             {installed_version("pip")}
- Notebook / Lab:  {installed_version("notebook")} / {installed_version("jupyterlab")}
- ipykernel:       {installed_version("ipykernel")}
- git:             {tool_version("git").split("   [")[0]}
- uv (optional):   {tool_version("uv").split("   [")[0]}
- Notes / surprises: <write yours here>
"""
print(summary)

---
## Your next step

| What Part B said | Do this next |
|---|---|
| **conda BASE environment** (most learners) | Follow [00-02-create-course-env.md](00-02-create-course-env.md) to create `lc-mastery` (≈10 minutes). Then open Lesson 00.03 and choose the **Python (lc-mastery)** kernel. |
| conda environment **'lc-mastery'** | You already have it. Go to Lesson 00.03. |
| A venv made with **uv** | Fine too. See the uv path in [the comparison guide](../../setup/PYTHON_ENVIRONMENTS_COMPARED.md#uv-one-fast-tool-for-almost-everything), and register the kernel as `lc-mastery`. |
| Python older than 3.11 | Leave it. The course env brings its own Python 3.13. |

---
## 🧰 Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `conda: command not found` in Git Bash | Anaconda does not add conda to Git Bash by default | `source ~/anaconda3/etc/profile.d/conda.sh` (this session) or `conda init bash` once ([VERIFY_ENVIRONMENT.md](../../setup/VERIFY_ENVIRONMENT.md#make-conda-work-in-git-bash)) |
| Typing `python` opens the **Microsoft Store** | Windows "App execution aliases" for python.exe | *Settings → Apps → Advanced app settings → App execution aliases*: turn off `python.exe` and `python3.exe`. Then use conda's Python |
| `which -a python` shows several Pythons | Several installs on the PATH (Store, python.org, Anaconda) | Normal. The **first** one wins. Activating a conda env puts its Python first |
| Notebook shows "No Kernel" or "Kernel error" | The kernelspec points to a deleted env | *Kernel → Change Kernel…* and pick another; remove stale ones with `jupyter kernelspec remove <name>` |
| Interactive `python` hangs in Git Bash | Old Git for Windows (before v2.53) needed `winpty` | Update Git for Windows ([release notes](https://github.com/git-for-windows/build-extra/blob/HEAD/ReleaseNotes.md)), or use `winpty python` on old versions |

### Windows paths in Git Bash: two spellings, one folder

| Windows (`cmd`, Python output) | Git Bash |
|---|---|
| `C:\Users\Ana\Documents` | `/c/Users/Ana/Documents` |
| `%USERPROFILE%` | `~` or `$HOME` |
| `C:\Users\Ana\anaconda3\Scripts\conda.exe` | `~/anaconda3/Scripts/conda.exe` |
| `cd "C:\My Docs\Course notes"` | `cd "/c/My Docs/Course notes"` (quotes needed for spaces) |

Convert between them in Git Bash with `cygpath -w /c/Users` and `cygpath -u 'C:\Users'` ([MSYS2 filesystem paths](https://www.msys2.org/docs/filesystem-paths/)).

---
## ⚠️ Production notes

> ⚠️ **Production note: your laptop environment is not a deployment.**
> A conda env built by hand is hard to reproduce. In production: pin exact versions in a lock file, build in CI from a clean image, and ship a container. See [Python environments, compared](../../setup/PYTHON_ENVIRONMENTS_COMPARED.md#recommendation-for-this-course) and Module 12 (Lesson 12.06).

## 🏋️ Exercises

1. **(Easy)** In Git Bash, run `which -a python` and `where python`. Do they list the same files in the same order? (`which` is Bash's lookup; `where` is the Windows one.)
2. **(Easy)** Add `"requests"` and `"numpy"` to `PACKAGES` and rerun Part C.
3. **(Medium)** Change `tool_version` so it also returns how long the command took (use `time.perf_counter()`). Which tool is slowest to start?
4. **(Stretch)** Write `kernel_python(name)` that reads the `kernel.json` of a kernelspec and returns the Python path in its `argv[0]`. Use it on `python3`.

<details>
<summary><b>Show solutions</b></summary>

**1.** Usually yes on Windows; `where` also searches the current folder first, `which` does not.

**4.**
```python
import json
def kernel_python(name: str) -> str:
    folder = KernelSpecManager().find_kernel_specs()[name]
    spec = json.loads((Path(folder) / "kernel.json").read_text(encoding="utf-8"))
    return spec["argv"][0]   # Often "python" (relative) for the default kernel, a full path for registered envs.
print(kernel_python("python3"))
```
</details>

## ✅ Checkpoint

- [ ] Git Bash prints versions for `python`, `git`, and (after sourcing `conda.sh`) `conda`.
- [ ] You know which environment this notebook's kernel uses (Part B).
- [ ] You pasted your environment summary into `docs/research-log.md`.
- [ ] You know your next step from the table above.

## 🔎 Research checkpoint

Open [Python versions status](https://devguide.python.org/versions/). Which Python versions are in "bugfix" and which in "security" status today? Is 3.13 still receiving bug fixes? Log it in `docs/research-log.md`.

## Summary

- The kernel, not the terminal, decides which Python runs your notebook. Check with `sys.executable`.
- In Git Bash, make conda available with `source ~/anaconda3/etc/profile.d/conda.sh` or `conda init bash`.
- `%pip` installs into the kernel's environment; terminal `pip` installs into whatever is first on the PATH.
- On Windows, `!command` in a notebook runs in `cmd.exe`, not Git Bash.

**Next:** create the course environment with [00-02-create-course-env.md](00-02-create-course-env.md), then open [00.03 · Jupyter fundamentals](00-03-jupyter-fundamentals.ipynb) with the **Python (lc-mastery)** kernel.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 00.03 · Jupyter fundamentals** → [`00-03-jupyter-fundamentals.ipynb`](00-03-jupyter-fundamentals.ipynb).

[Module 00 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)